# 实验作业五：SmolVLM-256M 的 QLoRA 与 DPO 对齐

本 Notebook 按实验指南记录：基线推理 → QLoRA SFT → 偏好对检查 → DPO → 三阶段评测。每个结论只在对应单元实际运行后填写。运行目录应为本 Notebook 所在的 `labs/05-smolvlm-qlora-dpo/`。

## 0. 环境与 GPU 自检

预期：CUDA 可用，且 `peft`、`accelerate`、`bitsandbytes` 可导入。若此处失败，停止训练并先记录报错。

In [3]:
import sys
import torch, transformers, peft, accelerate, bitsandbytes

print('Python:', sys.executable)
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Transformers:', transformers.__version__)
print('PEFT:', peft.__version__, '| Accelerate:', accelerate.__version__)
print('bitsandbytes:', bitsandbytes.__version__)

Python: <local Python 3.13 interpreter>
PyTorch: 2.6.0+cu124 | CUDA: 12.4
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Transformers: 5.4.0
PEFT: 0.21.2 | Accelerate: 1.15.0
bitsandbytes: 0.50.2


## 1. 生成固定种子的 15 张几何图形

图像由代码生成，随机种子固定为 42；它们位于 `hw5_imgs/`。

In [4]:
from pathlib import Path
import subprocess

ROOT = Path.cwd()
SCRIPT = ROOT / 'hw5_dpo.py'
assert SCRIPT.exists(), f'请从实验目录打开 Notebook：{SCRIPT}'
def run(*args):
    return subprocess.run([sys.executable, str(SCRIPT), *map(str, args)], check=True)

run('--mode', 'make_data')

CompletedProcess(args=['python', 'hw5_dpo.py', '--mode', 'make_data'], returncode=0)

## 2. 基线推理（必做）

首次运行会下载 `HuggingFaceTB/SmolVLM-256M-Instruct`。逐题输出即为基线证据：记录总体及 count/attr/exist 三类正确率；从输出中各挑选两例数错或存在性幻觉，原文留待报告引用。

In [4]:
run('--mode', 'infer')

CompletedProcess(args=['python', 'hw5_dpo.py', '--mode', 'infer'], returncode=0)

## 3. QLoRA SFT（必做）

CUDA 环境下，这一步使用 NF4 4-bit 基座和 LoRA `r=8, α=16, dropout=0.05`。输出中的 `trainable params` 比例需要抄入报告。默认 60 步；如发生显存问题，只可降低步数并如实记录。

In [5]:
run('--mode', 'sft', '--steps', '60')

CompletedProcess(args=['python', 'hw5_dpo.py', '--mode', 'sft', '--steps', '60'], returncode=0)

SFT 曲线写入 `results/sft_curve.json`，适配器写入 `sft_adapter/`。下面先检查曲线是否确实来自本次运行。

In [5]:
import json
sft_curve = json.loads((ROOT / 'results' / 'sft_curve.json').read_text(encoding='utf-8'))
print('SFT steps:', len(sft_curve))
print('first:', sft_curve[0])
print('last:', sft_curve[-1])

SFT steps: 60
first: {'step': 1, 'loss': 8.667640686035156}
last: {'step': 60, 'loss': 0.7946206331253052}


## 4. 偏好对与 DPO 手算（必做）

脚本包含指南给出的 12 组偏好对及 5 组自拟对；每个 rejected 均标注失败模式。先复核指南示例，再完成报告中的另外两组自选手算。注意：公式的精确值 `-log(sigmoid(0.01))` 为约 0.6882。

In [6]:
run('--dpo_check', '1.2,0.8,0.9,0.6,0.1')
# 思考题第 2 题给定数据：
run('--dpo_check', '1.5,1.0,0.2,0.6,0.1')
# 自选一组，改为自己手算的五个数后再运行：
run('--dpo_check', '0.9,0.7,0.4,0.6,0.1')

CompletedProcess(args=['python', 'hw5_dpo.py', '--dpo_check', '0.9,0.7,0.4,0.6,0.1'], returncode=0)

## 5. DPO 训练（必做）

DPO 从 SFT adapter 继续训练，参考模型为关闭 adapter 后的基座。记录 loss 和 margin：margin 由负转正或持续增大代表偏好方向被拉开，但不等于任务正确率一定提升。

In [ ]:
run('--mode', 'dpo', '--steps', '40', '--beta', '0.1')

In [7]:
dpo_curve = json.loads((ROOT / 'results' / 'dpo_curve.json').read_text(encoding='utf-8'))
print('DPO steps:', len(dpo_curve))
print('first:', dpo_curve[0])
print('last:', dpo_curve[-1])

DPO steps: 40
first: {'step': 1, 'loss': 0.8751434087753296, 'margin': -0.3359144330024719}
last: {'step': 40, 'loss': 0.5776644945144653, 'margin': 0.24606429040431976}


## 6. 三阶段 × 15 题评测（必做）

结果保存到 `results/evaluation.json`。报告表格应直接来自这里，并逐题人工抽取每阶段 3 题做 1–5 分忠实度评分（是否只陈述图中事实、是否有幻觉）。

In [8]:
run('--mode', 'eval')

CompletedProcess(args=['python', 'hw5_dpo.py', '--mode', 'eval'], returncode=0)

## 7. 结果整理与人工检查

下表由已保存的 `evaluation.json` 与逐题人工检查填写；自动正确率和人工忠实度采用不同口径。

| 阶段 | count /5 | attr /5 | exist /5 | 总计 /15 | 3 题人工忠实度 |
| --- | ---: | ---: | ---: | ---: | --- |
| 基线 | 1/5 | 0/5 | 0/5 | 1/15 | P01/2，P07/1，P12/5 |
| SFT | 5/5 | 2/5 | 2/5 | 9/15 | P01/5，P08/2，P12/1 |
| SFT+DPO | 5/5 | 1/5 | 3/5 | 9/15 | P01/5，P06/2，P14/1 |

可训练参数只读复核为 1,363,968 / 257,848,896 = 0.5290%。报告同时收录 SFT/DPO 曲线、3 组 DPO 手算、5 组自拟偏好对及失败模式、4 个真实失败案例和 6 道思考题。

## 8. 从已保存结果补充证据（不重跑训练）

以下单元只读取已完成训练写入的 `results/`、`sft_adapter/` 与 `dpo_adapter/`。它们不会重新训练、覆盖 adapter 或修改前面单元的 Output。运行后得到的图和表可直接用于报告。

In [ ]:
import json
import matplotlib.pyplot as plt

RESULTS = ROOT / 'results'
sft_curve = json.loads((RESULTS / 'sft_curve.json').read_text(encoding='utf-8'))
dpo_curve = json.loads((RESULTS / 'dpo_curve.json').read_text(encoding='utf-8'))
assert len(sft_curve) == 60 and len(dpo_curve) == 40, '曲线步数与本次实验记录不一致'

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot([x['step'] for x in sft_curve], [x['loss'] for x in sft_curve], color='#2563eb')
ax[0].set(title='SFT 训练损失', xlabel='训练步数', ylabel='Loss'); ax[0].grid(alpha=.25)
ax[1].plot([x['step'] for x in dpo_curve], [x['loss'] for x in dpo_curve], label='DPO loss', color='#dc2626')
ax[1].plot([x['step'] for x in dpo_curve], [x['margin'] for x in dpo_curve], label='margin', color='#16a34a')
ax[1].axhline(0, color='gray', lw=.8); ax[1].set(title='DPO 损失与偏好 margin', xlabel='训练步数'); ax[1].legend(); ax[1].grid(alpha=.25)
fig.tight_layout()
curve_path = RESULTS / 'training_curves.png'
fig.savefig(curve_path, dpi=180, bbox_inches='tight')
plt.show()
print('已由真实 JSON 生成：', curve_path)

In [ ]:
from collections import defaultdict
from IPython.display import display, Markdown

evaluation = json.loads((RESULTS / 'evaluation.json').read_text(encoding='utf-8'))
stages = [('baseline', '基线'), ('sft', 'SFT'), ('sft_dpo', 'SFT+DPO')]
summary = {}
for key, label in stages:
    rows = evaluation[key]
    summary[label] = {kind: sum(r['correct'] for r in rows if r['type'] == kind) for kind in ('count', 'attr', 'exist')}

display(Markdown('| 阶段 | count /5 | attr /5 | exist /5 | 总计 /15 |\n|---|---:|---:|---:|---:|\n' + '\n'.join(
    f"| {label} | {x['count']}/5 | {x['attr']}/5 | {x['exist']}/5 | {sum(x.values())}/15 |" for label, x in summary.items())))

# 每阶段完整逐题输出：从 evaluation.json 读取，供选取真实失败案例和人工忠实度评分。
for key, label in stages:
    display(Markdown(f'### {label}：逐题输出'))
    lines = ['|题号|类型|标准答案|模型输出|判定|', '|---|---|---|---|---|']
    for r in evaluation[key]:
        pred = r['pred'].replace('|', '\|').replace('\n', ' ')
        lines.append(f"|{r['id']}|{r['type']}|{r['gold']}|{pred}|{'√' if r['correct'] else '×'}|")
    display(Markdown('\n'.join(lines)))

## 9. QLoRA 可训练参数比例（只读复核）

该单元加载已经保存的 SFT adapter 并读取模型参数计数，不进行反向传播或 optimizer 更新。请截取输出中的总参数、可训练参数和比例，填入报告。

In [ ]:
from hw5_dpo import load_model, SFT_DIR

assert SFT_DIR.exists(), '未找到已完成的 SFT adapter'
processor_check, model_check = load_model(train=True, adapter=SFT_DIR, trainable=True)
trainable, total = model_check.get_nb_trainable_parameters()
print(f'总参数量: {total:,}')
print(f'可训练参数量: {trainable:,}')
print(f'可训练参数占比: {100 * trainable / total:.4f}%')
del processor_check, model_check
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 10. 人工忠实度评分记录（报告必填）

请从上方每阶段输出各选 3 题，按“是否仅陈述图中事实、有无幻觉”给 1–5 分。填写时保留题号与原始输出；不要根据正确率代替人工评分。

| 阶段 | 题号 1 / 分数 | 题号 2 / 分数 | 题号 3 / 分数 | 简要依据 |
|---|---|---|---|---|
| 基线 | P01 / 2 | P07 / 1 | P12 / 5 | P01 数错但未扩写；P07 将形状说成不存在的“白色”；P12 的英文 No. 与图中事实一致，虽未通过中文字符串匹配 |
| SFT | P01 / 5 | P08 / 2 | P12 / 1 | P01 正确且简短；P08 将方形误答圆形；P12 对不存在目标硬答“有”并重复标点 |
| SFT+DPO | P01 / 5 | P06 / 2 | P14 / 1 | P01 保持正确；P06 输出“外观”未回答形状；P14 对不存在的绿方形硬答“有” |

评分说明：忠实度评价图像事实，而不是仅按 `judge()` 的中文字符串判分；因此基线 P12 的英文 `No.` 语义正确，忠实度为 5 分。